In [5]:
import os
import ee
from typing import Literal
import sys

In [6]:
ee.Authenticate()
ee.Initialize()
sys.path.append('/data/leuven/378/vsc37883/SARPipeline/sandbox_zg')

from EeImageCollections.MultiOrbitCollection import MultiOrbitCollection
from EeImageCollections.SingleOrbitCollection import SingleOrbitCollection
from EeImageCollections.OPERA.OPERA_MultiOrbitCollection import OPERA_MultiOrbitCollection
from EeImageCollections.OPERA.OPERA_SingleOrbitCollection import OPERA_SingleOrbitCollection


In [7]:
assets = ee.FeatureCollection('projects/ee-zachariasguislain/assets/LeuvenFields')
roi = assets.bounds()
db_collection = OPERA_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'db')
linear_collection=  OPERA_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'linear')


In [8]:
db_37 = OPERA_SingleOrbitCollection.of(db_collection, 37)

In [9]:
print(db_37.get_bands())
print(db_37.get_relative_orbits())
print(db_37.get_relative_orbit())
print(db_37.get_pass_type())
print(db_37.get_time_period())
print(db_37.get_start_year())
print(db_37.get_relative_orbit_info())
print(db_37.get_collection_name())
print(db_37.get_collection_id())
print(db_37.get_footprint_size())
print(db_37.get_mother_collection_id())
print(db_37.get_units())
print(db_37 == db_37.of(db_collection, 37))


{'VH', 'VV'}
{37}
37
descending
('2024-01-01', '2024-12-31')
2024
{37: 'descending'}
OPERA/RTC/L2_V1/S1
59e2e439aae54c5c9083d110e8b0feea671d0c2288b298314f46085989408cbc
1700000
f210398832d228a0c3e2fa32e7fbdd5dcff091ad921582e665c8d69b4dc9c625
db
True


In [ ]:
db_37.add_db_cross_polarization()
print(db_37.get_bands())
cp_band = db_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

In [10]:
db_37.add_db_cross_polarization()
print(db_37.get_bands())
cp_band = db_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'CP', 'VH', 'VV'}
{'CP_max': 15.945073172493247, 'CP_min': -42.009478077845216}


In [12]:
db_collection = OPERA_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'db')
db_37 = OPERA_SingleOrbitCollection.of(db_collection, 37)
db_37.add_linear_cross_polarization()
print(db_37.get_bands())
cp_band = db_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'CP', 'VH', 'VV'}
{'CP_max': 39.31038680972646, 'CP_mean': 0.277710881546267, 'CP_min': 6.295818395786537e-05}


In [13]:
linear_37 = OPERA_SingleOrbitCollection.of(linear_collection, 37)
linear_37.add_db_cross_polarization()
print(linear_37.get_bands())
cp_band = linear_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'CP', 'VH', 'VV'}
{'CP_max': 15.945073172493247, 'CP_mean': -6.509362041320557, 'CP_min': -42.00947807784522}


In [14]:
linear_37 = OPERA_SingleOrbitCollection.of(linear_collection, 37)
linear_37.add_linear_cross_polarization()
print(linear_37.get_bands())
cp_band = linear_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'CP', 'VH', 'VV'}
{'CP_max': 39.31038680972647, 'CP_mean': 0.277710881546267, 'CP_min': 6.295818395786526e-05}


In [15]:
import geemap
my_map = geemap.Map()
my_map.centerObject(roi)
my_map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [23]:
my_map.addLayer(db_37._collection.select('VV').first(), {min:-25, max:-5}, name='VV (db)')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
linear_37.filter_border_noise()

vv_band = linear_37._collection.first().select('VV')
stats = vv_band.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=vv_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['VV'] >= 0.001)

angle_band = linear_37._collection.first().select('angle')
stats = angle_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=angle_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['angle_min'] > 28)
print(stats.getInfo()['angle_max'] < 47)

True


EEException: Image.select: Band pattern 'angle' did not match any bands. Available bands: [VV, VH, CP]

In [13]:
db_37.filter_border_noise()
my_map.addLayer(db_37.get_collection().first().select('VV'), name='VV (db) - border noise filtered')

vv_band = db_37._collection.first().select('VV')
stats = vv_band.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=vv_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['VV'] > -30)

angle_band = db_37._collection.first().select('angle')
stats = angle_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=angle_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['angle_min'] > 28)
print(stats.getInfo()['angle_max'] < 47)

True
True
True


In [14]:
db_37.add_azimuth_and_local_incidence_angle()
db_37.get_bands()

{'AZI', 'CP', 'LIA', 'VH', 'VV', 'angle'}

In [15]:
my_map.addLayer(db_37.get_collection().first().select('AZI'), name="azimuth")
my_map.addLayer(db_37.get_collection().first().select('LIA'), name="local incidence angle")

In [16]:
azi = db_37._collection.first().select('AZI')
stats = azi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=azi.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

stats

In [17]:
lia = db_37._collection.first().select('LIA')
stats = lia.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True).combine(ee.Reducer.stdDev(), sharedInputs = True),
    geometry=lia.geometry(),    # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

stats

In [24]:
db_37.get_name()

'OPERA_RO_37_Pass_des_Year_2024'

In [25]:
db_37.get_bands_per_reducer()

{'mean_count_stdev': set(), 'mean_stdev': set(), 'mean': set()}

In [26]:
db_37.get_reducers()

{'mean_count_stdev': <ee.reducer.Reducer at 0x150c2d76a250>,
 'mean_stdev': <ee.reducer.Reducer at 0x150c2d76a810>,
 'mean': <ee.reducer.Reducer at 0x150c2d76a490>}

In [27]:

print(db_37.get_batch_size() == 500)
db_37.set_batch_size(20)
print(db_37.get_batch_size() == 20)

True
True


In [28]:
db_37.get_mother_collection_id() == db_collection.get_collection_id()

True

In [29]:
old_reducers = db_37.get_reducers()
db_37.add_reducer("min_max", ee.Reducer.minMax())
new_reducers = db_37.get_reducers()
print(old_reducers)
print(new_reducers)

{'mean_count_stdev': <ee.reducer.Reducer object at 0x150c2d76a250>, 'mean_stdev': <ee.reducer.Reducer object at 0x150c2d76a810>, 'mean': <ee.reducer.Reducer object at 0x150c2d76a490>}
{'mean_count_stdev': <ee.reducer.Reducer object at 0x150c2d76a250>, 'mean_stdev': <ee.reducer.Reducer object at 0x150c2d76a810>, 'mean': <ee.reducer.Reducer object at 0x150c2d76a490>, 'min_max': <ee.reducer.Reducer object at 0x150c2689b410>}


In [32]:
db_37.allocate_band_to_reducer('mean_count_stdev', 'VV')
db_37.allocate_bands_to_reducer('mean_stdev', {'VH', 'CP'})
db_37.allocate_band_to_reducer('mean', 'AZI')

ValueError: band AZI is not present in the bands of this collection ({'CP', 'VH', 'VV'}).

In [33]:
db_37.get_variables_after_reduction()

{'CP_mean',
 'CP_stdDev',
 'VH_mean',
 'VH_stdDev',
 'VV_count',
 'VV_mean',
 'VV_stdDev'}

In [35]:
reduced_collection = db_37.reduce(assets, 'Name')

In [37]:
ds = reduced_collection.create_dataset()

KeyError: 'Name'